In [ ]:
import os, shutil, random
from pathlib import Path
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers

# Kaggle environment — outputs persist in /kaggle/working (20GB).
# In the right sidebar set:  Accelerator = GPU   and   Internet = ON
# (Internet is needed for the MobileNetV2 ImageNet weights download in Cell 8.)
DRIVE_BASE = '/kaggle/working'

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print("Setup complete")


In [ ]:
# Datasets attached via 'Add Data'. Paths hardcoded from the Input panel tree:
#   /kaggle/input/datasets/<owner>/<slug>/...
BASE = '/kaggle/input/datasets'

PLANTVILLAGE_BASE = f'{BASE}/abdallahalidev/plantvillage-dataset/color'
IP102_ROOT        = f'{BASE}/rtlmhjbn/ip02-dataset'
IP102_TRAIN_DIR   = f'{IP102_ROOT}/classification/train'
PLANTDOC_DIR      = f'{BASE}/nirmalsankalana/plantdoc-dataset'
MAIZE_DIR         = f'{BASE}/ashishpatelresearch/maize-plant-leaf-nutrient-deficiency-dataset'

print("Dataset paths:")
for label, p in [('PlantVillage color', PLANTVILLAGE_BASE),
                  ('IP102 root',        IP102_ROOT),
                  ('IP102 train',       IP102_TRAIN_DIR),
                  ('PlantDoc',          PLANTDOC_DIR),
                  ('Maize NPK',         MAIZE_DIR)]:
    print(f"  [{'OK' if os.path.exists(p) else 'MISSING'}] {label:<20} {p}")


In [ ]:
# Final 5 classes. Sources are MIXED per class on purpose:
# lab images (PlantVillage) + field images (PlantDoc) force the model
# off background cues instead of memorizing clean lab backdrops.
# Paths PLANTVILLAGE_BASE / IP102_TRAIN_DIR / PLANTDOC_DIR / MAIZE_DIR come from Cell 1.
OUTPUT_DIR = '/kaggle/working/dataset'

CLASSES = ['Early_Blight', 'Healthy', 'Late_Blight', 'Nutrient_Deficiency', 'Pest']

# --- PlantVillage: lab images for disease + healthy classes ---
PLANTVILLAGE_MAP = {
    'Healthy': [
        'Pepper,_bell___healthy',
        'Potato___healthy',
        'Tomato___healthy',
    ],
    'Early_Blight': [
        'Potato___Early_blight',
        'Tomato___Early_blight',
    ],
    'Late_Blight': [
        'Potato___Late_blight',
        'Tomato___Late_blight',
    ],
}
# NOTE: old 'Nutrient_Deficiency' = 4 tomato DISEASE folders (Leaf_Mold,
# Target_Spot, mosaic/curl virus). Those are diseases, NOT nutrient deficiency —
# mislabeled, and the high variety made it the model's "dump" class for
# anything uncertain. Dropped. Real nutrient data now comes from Maize NPK.

MAX_PER_CLASS = 2500

available = os.listdir(PLANTVILLAGE_BASE)
print(f"PlantVillage color folders: {len(available)}")
print("\nVerifying PlantVillage folders:")
for cls, folders in PLANTVILLAGE_MAP.items():
    for f in folders:
        print(f"  [{'OK' if f in available else 'MISSING'}] {cls:<14} {f}")
print(f"\nIP102 train dir exists: {os.path.isdir(IP102_TRAIN_DIR)}")


# --- helper: every dir that directly contains image files ---
# Returns a LIST of (name, path) — datasets with train/ AND test/ have
# same-named class folders; a dict keyed by name would silently drop one.
def find_image_folders(root):
    found = []
    for dirpath, _, filenames in os.walk(root):
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in filenames):
            found.append((os.path.basename(dirpath), dirpath))
    return found

DISEASE_WORDS = ['blight', 'spot', 'mold', 'mould', 'virus', 'rust', 'mildew',
                 'rot', 'bacterial', 'mosaic', 'curl', 'scab', 'deficien']

# --- PlantDoc: field images, keyword-matched into 3 disease/healthy classes ---
PLANTDOC_MAP = {'Healthy': [], 'Early_Blight': [], 'Late_Blight': []}
for name, path in find_image_folders(PLANTDOC_DIR):
    low = name.lower()
    if 'blight' in low and 'early' in low:
        PLANTDOC_MAP['Early_Blight'].append(path)
    elif 'blight' in low and 'late' in low:
        PLANTDOC_MAP['Late_Blight'].append(path)
    elif (any(c in low for c in ['tomato', 'potato', 'pepper', 'bell'])
          and not any(w in low for w in DISEASE_WORDS)):
        PLANTDOC_MAP['Healthy'].append(path)

print("\nPlantDoc matched folders (field images):")
for cls, paths in PLANTDOC_MAP.items():
    names = [os.path.basename(p) for p in paths]
    print(f"  {cls:<14} {len(paths)}: {names}")

# --- Maize NPK: deficiency folders -> Nutrient_Deficiency (skip healthy maize) ---
MAIZE_FOLDERS = [path for name, path in find_image_folders(MAIZE_DIR)
                 if 'healthy' not in name.lower()]

print("\nMaize NPK folders -> Nutrient_Deficiency:")
for p in MAIZE_FOLDERS:
    print(f"  {os.path.relpath(p, MAIZE_DIR)}")
print("\nReview matches above. If a folder is mis-bucketed, edit the keyword rules.")


In [ ]:
# Gather every source image per class, THEN shuffle + cap + copy once.
# Old code capped each source separately; this caps the merged pool so the
# lab/field mix inside a class is preserved.

for cls in CLASSES:
    os.makedirs(Path(OUTPUT_DIR) / cls, exist_ok=True)

def list_images(folder):
    p = Path(folder)
    return (list(p.glob('*.jpg')) + list(p.glob('*.JPG'))
            + list(p.glob('*.jpeg')) + list(p.glob('*.png')))

class_images = {cls: [] for cls in CLASSES}

# --- PlantVillage (lab) ---
for cls, folders in PLANTVILLAGE_MAP.items():
    for folder in folders:
        src = Path(PLANTVILLAGE_BASE) / folder
        if src.exists():
            class_images[cls].extend(list_images(src))
        else:
            print(f"WARN: PlantVillage folder missing: {folder}")

# --- PlantDoc (field) ---
for cls, paths in PLANTDOC_MAP.items():
    for path in paths:
        class_images[cls].extend(list_images(path))

# --- Maize NPK -> Nutrient_Deficiency ---
for path in MAIZE_FOLDERS:
    class_images['Nutrient_Deficiency'].extend(list_images(path))

# --- IP102 -> Pest ---
ip102_lookup = {}
for class_folder in os.listdir(IP102_TRAIN_DIR):
    class_path = Path(IP102_TRAIN_DIR) / class_folder
    if class_path.is_dir():
        for img in class_path.glob('*.jpg'):
            ip102_lookup[img.name] = img
with open(f'{IP102_ROOT}/train.txt') as f:
    for line in f:
        parts = line.strip().split()
        if parts and parts[0] in ip102_lookup:
            class_images['Pest'].append(ip102_lookup[parts[0]])

# --- shuffle, cap, copy ---
print(f"\n=== Copying (cap {MAX_PER_CLASS}/class) ===")
copied = {}
for cls in CLASSES:
    imgs = class_images[cls]
    random.shuffle(imgs)
    imgs = imgs[:MAX_PER_CLASS]
    n = 0
    for i, img in enumerate(imgs):
        # numeric prefix avoids filename collisions across the 4 datasets
        dst = Path(OUTPUT_DIR) / cls / f"{i:05d}_{img.name}"
        if not dst.exists():
            shutil.copy(img, dst)
            n += 1
    copied[cls] = n
    bar = '#' * (n // 100)
    print(f"  {cls:<22} {n:>5}  {bar}")

total = sum(copied.values())
print(f"\nTotal: {total} images")
for cls in CLASSES:
    if copied[cls] == 0:
        print(f"  ERROR: {cls} has 0 images — check dataset mapping in Cell 2")


In [ ]:

classes = []
counts  = []

for cls in sorted(os.listdir(OUTPUT_DIR)):
    path = Path(OUTPUT_DIR) / cls

    if not path.is_dir():
        continue

    n = len([
        f for f in path.rglob("*")
        if f.is_file()
    ])

    classes.append(cls.replace('_', '\n'))
    counts.append(n)

fig, ax = plt.subplots(figsize=(10, 4))

bars = ax.bar(
    classes,
    counts,
    color='#2d9e5f',
    edgecolor='#0a2e1a',
    linewidth=0.5
)

ax.set_title('Dataset class balance', fontsize=13, pad=12)
ax.set_ylabel('Images')

if counts:
    ax.set_ylim(0, max(counts) * 1.15)

for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 50,
        str(count),
        ha='center',
        va='bottom',
        fontsize=10
    )

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/class_balance.png', dpi=120)
plt.show()

print("Chart saved to Drive")

In [ ]:

fig, axes = plt.subplots(len(os.listdir(OUTPUT_DIR)), 3, figsize=(9, 14))
fig.suptitle('Sample images per class', fontsize=13, y=1.01)

for row, cls in enumerate(sorted(os.listdir(OUTPUT_DIR))):
    imgs = [p for p in Path(f'{OUTPUT_DIR}/{cls}').iterdir()
        if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
    samples = random.sample(imgs, min(3, len(imgs)))
    for col, img_path in enumerate(samples):
        img = Image.open(img_path).resize((224, 224))
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_title(cls.replace('_', ' '),
                                     fontsize=9, loc='left', pad=3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/sample_images.png', dpi=100, bbox_inches='tight')
plt.show()
print("Samples saved to Drive")

In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42  # CRITICAL: Seed must be identical for both splits
NUM_CLASSES = 5

# 1. Load ONLY the Training Set (80%)
train_ds = tf.keras.utils.image_dataset_from_directory(
    OUTPUT_DIR,
    validation_split=0.2, # 20% reserved for val/test
    subset="training",    # Pull the training portion
    seed=SEED,            # Seed ensures the split is identical
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

# 2. Load the Validation/Test Pool (The remaining 20%)
val_test_pool = tf.keras.utils.image_dataset_from_directory(
    OUTPUT_DIR,
    validation_split=0.2,
    subset="validation",  # Pull the unseen portion
    seed=SEED,            # Must exactly match the train_ds seed!
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

# 3. Split the 20% pool exactly in half to get 10% Val / 10% Test
# Because shuffle=True by default in the loader, we can safely take/skip here
# WITHOUT reshuffling issues, as long as we don't re-iterate the base pool randomly.
val_batches = len(val_test_pool) // 2
val_ds  = val_test_pool.take(val_batches)
test_ds = val_test_pool.skip(val_batches)

print(f"Train: {len(train_ds)} batches")
print(f"Val:   {len(val_ds)} batches")
print(f"Test:  {len(test_ds)} batches")

# 4. Performance Tuning
AUTOTUNE = tf.data.AUTOTUNE

# Add an explicit .shuffle() to train_ds AFTER caching.
# This shuffles the batches dynamically in memory every epoch,
# which is vastly superior for preventing overfitting.
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)

# Validation and Test should NOT be shuffled
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

In [ ]:
augmentation = tf.keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
  layers.RandomBrightness(0.25, value_range=(0, 255)),  # Pi camera + indoor light vary
  layers.RandomContrast(0.2),
  layers.RandomTranslation(0.1, 0.1),
], name='augmentation')

# Preview augmented images to verify it looks reasonable
sample_images, _ = next(iter(train_ds.take(1)))
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Original (top) vs Augmented (bottom)', fontsize=11)
for i in range(5):
    img = sample_images[i].numpy().astype('uint8')
    aug = augmentation(tf.expand_dims(sample_images[i], 0), training=True)
    aug = aug[0].numpy().clip(0, 255).astype('uint8')
    axes[0][i].imshow(img);  axes[0][i].axis('off')
    axes[1][i].imshow(aug);  axes[1][i].axis('off')
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/augmentation_preview.png', dpi=100)
plt.show()


In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.applications import MobileNetV2

# Load base model — weights from ImageNet, no top classifier
base = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)

# Freeze first 100 layers — keep low-level ImageNet features
base.trainable = True
for layer in base.layers[:100]:
    layer.trainable = False

frozen  = sum(1 for l in base.layers if not l.trainable)
unfrozen = sum(1 for l in base.layers if l.trainable)
print(f"Base model: {frozen} layers frozen, {unfrozen} layers trainable")

# Build full model
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augmentation(inputs, training=True)          # augmentation in-graph
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)  # scale to [-1,1]
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.6)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs, name='irrigation_mobilenetv2')
model.summary(show_trainable=True)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get class names from the output directory, ensuring sorted order
CLASS_NAMES = sorted(os.listdir(OUTPUT_DIR))

# Collect all labels from train set
all_labels = []
for _, labels in train_ds.unbatch():
    all_labels.append(np.argmax(labels.numpy()))

all_labels = np.array(all_labels)
unique_classes = np.unique(all_labels)

weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=all_labels,
)

class_weight_dict = dict(zip(unique_classes, weights))
print("Class weights:")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name:<22}: {class_weight_dict[idx]:.3f}")

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)

CHECKPOINT_PATH = f'{DRIVE_BASE}/best_model.keras'

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks = [
    # Stop early if val_accuracy stops improving
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    # Save best model to Drive automatically
    ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
    ),
    # Reduce learning rate if stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

print("Starting training...")
print(f"Best model will be saved to: {CHECKPOINT_PATH}")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'],     label='Train', color='#2d9e5f', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Val',   color='#0a2e1a', linewidth=2, linestyle='--')
ax1.set_title('Accuracy over epochs')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_ylim(0, 1)

ax2.plot(history.history['loss'],     label='Train', color='#e8a020', linewidth=2)
ax2.plot(history.history['val_loss'], label='Val',   color='#d94040', linewidth=2, linestyle='--')
ax2.set_title('Loss over epochs')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/training_curves.png', dpi=120)
plt.show()

final_acc = history.history['val_accuracy'][-1]
best_acc  = max(history.history['val_accuracy'])
print(f"Final val accuracy: {final_acc:.3f}")
print(f"Best  val accuracy: {best_acc:.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5)
ax.set_title('Confusion matrix (row-normalized)', fontsize=12, pad=12)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/confusion_matrix.png', dpi=120)
plt.show()

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
# --- Entropy calibration: pick ENTROPY_THRESHOLD for the Unknown fallback ---
# Softmax always sums to 1, so an off-distribution leaf gets FORCED into a
# class — that is why the old model "always said Nutrient_Deficiency".
# Fix: when the softmax is smeared across classes (high normalized entropy),
# treat the prediction as Unknown instead of trusting argmax.
import json

entropies, correct = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    for p, lbl in zip(preds, labels.numpy()):
        p   = np.clip(p, 1e-9, 1.0)
        ent = -np.sum(p * np.log(p)) / np.log(len(CLASS_NAMES))  # normalized 0..1
        entropies.append(ent)
        correct.append(int(np.argmax(p) == np.argmax(lbl)))

entropies = np.array(entropies)
correct   = np.array(correct)
ent_correct = entropies[correct == 1]
ent_wrong   = entropies[correct == 0]

# Threshold = 95th percentile of CORRECT-prediction entropy:
# correct preds are rarely flagged, smeared/wrong preds get caught as Unknown.
ENTROPY_THRESHOLD = float(np.percentile(ent_correct, 95)) if len(ent_correct) else 0.55

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ent_correct, bins=30, alpha=0.7, label='correct', color='#2d9e5f')
ax.hist(ent_wrong,   bins=30, alpha=0.7, label='wrong',   color='#d94040')
ax.axvline(ENTROPY_THRESHOLD, color='k', linestyle='--',
           label=f'threshold {ENTROPY_THRESHOLD:.3f}')
ax.set_title('Prediction entropy: correct vs wrong')
ax.set_xlabel('Normalized entropy'); ax.set_ylabel('Count'); ax.legend()
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/entropy_calibration.png', dpi=120)
plt.show()

flagged = (ent_wrong > ENTROPY_THRESHOLD).mean() * 100 if len(ent_wrong) else 0
print(f"ENTROPY_THRESHOLD = {ENTROPY_THRESHOLD:.4f}")
print(f"  correct preds: mean entropy {ent_correct.mean():.3f}")
print(f"  wrong   preds: mean entropy {ent_wrong.mean():.3f}  "
      f"({flagged:.0f}% of wrong preds would be flagged Unknown)")
print("--> Copy this ENTROPY_THRESHOLD into pi_daemon.py CONFIG")

with open(f'{DRIVE_BASE}/entropy_threshold.json', 'w') as f:
    json.dump({'entropy_threshold': ENTROPY_THRESHOLD}, f)


In [ ]:
import json

# Save class names so anyone loading the model knows the label order
with open(f'{DRIVE_BASE}/class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f)

# Save training history for later analysis
with open(f'{DRIVE_BASE}/training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals]
               for k, vals in history.history.items()}, f)

# Best model is already saved by ModelCheckpoint to best_model.keras
# Also save final model
model.save(f'{DRIVE_BASE}/final_model.keras')

print("Saved to Drive:")
print(f"  {DRIVE_BASE}/best_model.keras")
print(f"  {DRIVE_BASE}/final_model.keras")
print(f"  {DRIVE_BASE}/class_names.json")
print(f"  {DRIVE_BASE}/training_history.json")
print(f"  {DRIVE_BASE}/confusion_matrix.png")
print(f"  {DRIVE_BASE}/training_curves.png")

In [ ]:
pip install ai-edge-litert

In [ ]:
import gc
import os
import json
import numpy as np
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

DRIVE_BASE  = '/kaggle/working'
OUTPUT_DIR  = '/kaggle/working/dataset'
MODEL_PATH  = f'{DRIVE_BASE}/best_model.keras'
TFLITE_PATH = f'{DRIVE_BASE}/irrigation_int8.tflite'
IMG_SIZE    = 224
BATCH_SIZE  = 32
SEED        = 42

In [ ]:
val_test_pool = tf.keras.utils.image_dataset_from_directory(
    OUTPUT_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

val_batches = len(val_test_pool) // 2
test_ds  = val_test_pool.skip(val_batches).prefetch(tf.data.AUTOTUNE)
calib_ds = val_test_pool.take(val_batches).prefetch(tf.data.AUTOTUNE)

model = tf.keras.models.load_model(MODEL_PATH)
_, keras_acc = model.evaluate(test_ds, verbose=1)
print(f"\nKeras float32 accuracy: {keras_acc:.4f}  ({keras_acc*100:.2f}%)")

del model
gc.collect()
tf.keras.backend.clear_session()
print("Keras model freed from memory")

In [ ]:
CALIB_BATCHES = 20

def representative_dataset():
    count = 0
    for images, _ in calib_ds.take(CALIB_BATCHES):
        for img in images:
            yield [tf.expand_dims(img, axis=0)]
            count += 1
    print(f"Calibration samples: {count}")

model = tf.keras.models.load_model(MODEL_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations             = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type      = tf.int8
converter.inference_output_type     = tf.int8

tflite_model = converter.convert()

del model
gc.collect()
tf.keras.backend.clear_session()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_PATH) / 1e6
print(f"Saved : {TFLITE_PATH}")
print(f"Size  : {size_mb:.2f} MB  {'✓ PASS (<4 MB)' if size_mb < 4 else '✗ FAIL (>=4 MB)'}")

In [ ]:
interpreter = Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

inp  = interpreter.get_input_details()[0]
out  = interpreter.get_output_details()[0]
in_scale,  in_zp  = inp['quantization']
out_scale, out_zp = out['quantization']

print(f"Input  — scale={in_scale:.6f}  zero_point={in_zp}")
print(f"Output — scale={out_scale:.6f}  zero_point={out_zp}")

correct, total = 0, 0

for images, labels in test_ds:
    for img, label in zip(images.numpy(), labels.numpy()):
        img_int8 = np.round(img / in_scale + in_zp).clip(-128, 127).astype(np.int8)
        interpreter.set_tensor(inp['index'], img_int8[np.newaxis])
        interpreter.invoke()
        logits = (interpreter.get_tensor(out['index']).astype(np.float32) - out_zp) * out_scale
        correct += int(np.argmax(logits) == np.argmax(label))
        total   += 1

tflite_acc = correct / total
drop       = keras_acc - tflite_acc

print(f"\nKeras  accuracy : {keras_acc:.4f}  ({keras_acc*100:.2f}%)")
print(f"TFLite accuracy : {tflite_acc:.4f}  ({tflite_acc*100:.2f}%)")
print(f"Accuracy drop   : {drop*100:.2f} pp  {'✓ PASS (<2pp)' if drop < 0.02 else '✗ FAIL (>=2pp)'}")

In [ ]:
with open(f'{DRIVE_BASE}/model.tflite', 'wb') as f:
    f.write(tflite_model)

summary = {
    'keras_accuracy' : float(keras_acc),
    'tflite_accuracy': float(tflite_acc),
    'accuracy_drop'  : float(drop),
    'model_size_mb'  : float(size_mb),
    'in_scale'       : float(in_scale),
    'in_zero_point'  : int(in_zp),
    'out_scale'      : float(out_scale),
    'out_zero_point' : int(out_zp),
    'img_size'       : IMG_SIZE,
    'num_classes'    : 5,
}
with open(f'{DRIVE_BASE}/tflite_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Model saved   → {DRIVE_BASE}/model.tflite")
print(f"Summary saved → {DRIVE_BASE}/tflite_summary.json")

In [ ]:
import struct
import time
from dataclasses import dataclass
from enum import IntEnum
from typing import Optional

# Per proposal Section 5.2
START_BYTE = 0xBB
END_BYTE   = 0xEE
PACKET_LEN = 8

CLASS_NAMES = ['Early_Blight', 'Healthy', 'Late_Blight',
               'Nutrient_Deficiency', 'Pest', 'Unknown']

# Per proposal Section 5.1
CLASS_TO_PROTOCOL = {
    0: 0x02,  # Early_Blight       → REDUCE_IRR
    1: 0x01,  # Healthy            → CONTINUE_NORMAL
    2: 0x03,  # Late_Blight        → EMERGENCY_DRY
    3: 0x05,  # Nutrient_Deficiency→ INCREASE_IRR
    4: 0x04,  # Pest               → TARGETED_SPRAY
    5: 0xFF,  # Unknown            → FLAG_FOR_REVIEW
}

CONFIDENCE_THRESHOLDS = {
    0: 0.80,  # Early_Blight
    1: 0.90,  # Healthy
    2: 0.75,  # Late_Blight
    3: 0.75,  # Nutrient_Deficiency
    4: 0.70,  # Pest
    5: 0.00,  # Unknown
}

# Param1, Param2 per protocol
PROTOCOL_PARAMS = {
    0x01: (0,   1),   # CONTINUE_NORMAL
    0x02: (60,  1),   # REDUCE_IRR: 60% duty
    0x03: (0,   1),   # EMERGENCY_DRY: pump off
    0x04: (3,   6),   # TARGETED_SPRAY: 3s burst, 6x freq
    0x05: (130, 1),   # INCREASE_IRR: 130% duty
    0xFF: (0,   1),   # FLAG_REVIEW: no action
}

@dataclass
class Packet:
    protocol_id : int
    zone_x      : int
    param1      : int
    param2      : int
    raw         : bytes

    def __repr__(self):
        return (f"Packet(proto=0x{self.protocol_id:02X}, "
                f"zone_x={self.zone_x}, "
                f"raw={self.raw.hex(' ').upper()})")

def build_packet(class_index: int, confidence: float, zone_x: int = 0x80) -> Packet:
    # Validate class
    if class_index < 0 or class_index >= len(CLASS_NAMES):
        class_index = 5  # Unknown

    # Check confidence threshold
    threshold = CONFIDENCE_THRESHOLDS.get(class_index, 0.70)
    if confidence < threshold:
        protocol_id = 0xFF  # FLAG_FOR_REVIEW
    else:
        protocol_id = CLASS_TO_PROTOCOL[class_index]

    param1, param2 = PROTOCOL_PARAMS[protocol_id]
    zone_y = 0x00  # X-axis gantry only

    # [0xBB][PROTO_ID][ZONE_X][ZONE_Y][P1][P2][CRC][0xEE]
    payload = bytes([START_BYTE, protocol_id, zone_x, zone_y, param1, param2])
    crc = 0
    for b in payload[1:]:  # CRC over bytes 1-5
        crc ^= b
    raw = payload + bytes([crc & 0xFF, END_BYTE])

    return Packet(protocol_id, zone_x, param1, param2, raw)

def verify_packet(raw: bytes) -> bool:
    if len(raw) != PACKET_LEN:         return False
    if raw[0] != START_BYTE:           return False
    if raw[-1] != END_BYTE:            return False
    crc = 0
    for b in raw[1:-2]:
        crc ^= b
    return crc == raw[-2]

# Demo
print("Protocol demo:")
for label, cls, conf in [
    ("Healthy, high conf",              1, 0.92),
    ("Early Blight, high conf",         0, 0.85),
    ("Late Blight, high conf",          2, 0.80),
    ("Nutrient Deficiency, high conf",  3, 0.77),
    ("Pest, high conf",                 4, 0.96),
    ("Any class, below threshold",      1, 0.60),
    ("Invalid class",                  99, 0.99),
]:
    pkt = build_packet(cls, conf)
    valid = verify_packet(pkt.raw)
    print(f"  {label:<38} → {pkt}  checksum={'✓' if valid else '✗'}")

In [ ]:
import unittest

class TestProtocol(unittest.TestCase):

    def _pkt(self, cls, conf, zone_x=0x80):
        p = build_packet(cls, conf, zone_x)
        self.assertTrue(verify_packet(p.raw),  f"Bad packet: {p}")
        self.assertEqual(len(p.raw), 8,         f"Wrong length: {p}")
        self.assertEqual(p.raw[0], START_BYTE,  f"Wrong start byte: {p}")
        self.assertEqual(p.raw[-1], END_BYTE,   f"Wrong end byte: {p}")
        return p

    def test_healthy_high_conf(self):
        self.assertEqual(self._pkt(1, 0.92).protocol_id, 0x01)

    def test_early_blight_high_conf(self):
        self.assertEqual(self._pkt(0, 0.85).protocol_id, 0x02)

    def test_late_blight_high_conf(self):
        self.assertEqual(self._pkt(2, 0.80).protocol_id, 0x03)

    def test_nutrient_def_high_conf(self):
        self.assertEqual(self._pkt(3, 0.77).protocol_id, 0x05)

    def test_pest_high_conf(self):
        self.assertEqual(self._pkt(4, 0.96).protocol_id, 0x04)

    def test_below_threshold_flags_review(self):
        self.assertEqual(self._pkt(1, 0.60).protocol_id, 0xFF)

    def test_invalid_class_flags_review(self):
        self.assertEqual(self._pkt(99, 0.99).protocol_id, 0xFF)

    def test_zone_y_always_zero(self):
        self.assertEqual(self._pkt(1, 0.92).raw[3], 0x00)

    def test_all_packets_8_bytes(self):
        for cls, conf in [(1,.92),(0,.85),(2,.80),(3,.77),(4,.96),(1,.60)]:
            self.assertEqual(len(build_packet(cls, conf).raw), 8)

    def test_checksum_all_protocols(self):
        for cls, conf in [(1,.92),(0,.85),(2,.80),(3,.77),(4,.96),(1,.60)]:
            self.assertTrue(verify_packet(build_packet(cls, conf).raw))

    def test_corrupt_detected(self):
        raw = bytearray(build_packet(1, 0.92).raw)
        raw[3] ^= 0xFF
        self.assertFalse(verify_packet(bytes(raw)))

suite  = unittest.TestLoader().loadTestsFromTestCase(TestProtocol)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{'ALL TESTS PASSED ✓' if result.wasSuccessful() else 'FAILURES DETECTED ✗'}")

In [ ]:
daemon_code = '''#!/usr/bin/env python3
"""
Pi Inference Daemon — Smart Precision Irrigation System
Captures images -> runs TFLite inference -> sends UART protocol to PIC -> pushes to Firebase.

Usage:
    python3 pi_daemon.py                       # Normal operation with camera
    python3 pi_daemon.py --test-image leaf.jpg # Test with a static image
"""

import argparse
import logging
import signal
import sys
import time
from datetime import datetime, timezone

import numpy as np
import serial
import firebase_admin
from firebase_admin import credentials, db
from ai_edge_litert.interpreter import Interpreter

# ─── CONFIG ──────────────────────────────────────────────────────────────
UART_PORT        = "/dev/ttyAMA0"
UART_BAUD        = 9600
MODEL_PATH       = "/home/akai2004/irrigation/model.tflite"
FIREBASE_KEY     = "/home/akai2004/irrigation/firebase-service-key.json"
FIREBASE_DB      = "https://embedded-project-32dca-default-rtdb.firebaseio.com"
LOG_PATH         = "/tmp/irrigation_daemon.log"

START_BYTE       = 0xBB
END_BYTE         = 0xEE

INPUT_SIZE       = 224
CAPTURE_INTERVAL = 1.0

CLASS_NAMES = ["Early_Blight", "Healthy", "Late_Blight", "Nutrient_Deficiency", "Pest"]

# Unknown fallback: a prediction is Unknown if EITHER
#   max-prob < per-class threshold   OR   normalized entropy > ENTROPY_THRESHOLD.
# Entropy catches "confident-looking but smeared" softmax outputs that the
# max-prob check alone misses. Set from the notebook entropy-calibration cell.
ENTROPY_THRESHOLD = 0.55

PROTOCOL_MAP = {
    "Early_Blight":        0x02,
    "Healthy":             0x01,
    "Late_Blight":         0x03,
    "Nutrient_Deficiency": 0x05,
    "Pest":                0x04,
}

CONFIDENCE_THRESHOLDS = {
    "Early_Blight":        0.80,
    "Healthy":             0.90,
    "Late_Blight":         0.75,
    "Nutrient_Deficiency": 0.75,
    "Pest":                0.70,
}

PROTOCOL_PARAMS = {
    0x01: (0,   1),
    0x02: (60,  1),
    0x03: (0,   1),
    0x04: (3,   6),
    0x05: (130, 1),
    0xFF: (0,   1),
}

MAX_RETRIES  = 3
RETRY_DELAY  = 0.5

# ─── LOGGING ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("pi_daemon")

# ─── GRACEFUL SHUTDOWN ───────────────────────────────────────────────────
shutdown_flag = False

def signal_handler(sig, frame):
    global shutdown_flag
    logger.info(f"Received signal {sig}, shutting down...")
    shutdown_flag = True

signal.signal(signal.SIGTERM, signal_handler)
signal.signal(signal.SIGINT, signal_handler)

# ─── UART ────────────────────────────────────────────────────────────────
def build_packet(protocol_id: int, zone_x: int, param1: int, param2: int) -> bytes:
    zone_y  = 0x00
    payload = bytes([START_BYTE, protocol_id, zone_x, zone_y, param1, param2])
    crc     = 0
    for b in payload[1:]:
        crc ^= b
    return payload + bytes([crc & 0xFF, END_BYTE])

def send_packet(ser: serial.Serial, packet: bytes) -> bool:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            n = ser.write(packet)
            ser.flush()
            if n == len(packet):
                logger.info(f"UART TX OK (attempt {attempt}): {packet.hex()}")
                return True
            logger.warning(f"Partial write: {n}/{len(packet)}")
        except serial.SerialException as e:
            logger.error(f"UART error (attempt {attempt}): {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    logger.error(f"UART TX FAILED: {packet.hex()}")
    return False

# ─── FIREBASE ────────────────────────────────────────────────────────────
def push_to_firebase(cls: str, conf: float, entropy: float, proto: int, zone_x: int):
    try:
        db.reference("/ai/latest_detection").set({
            "class":      cls,
            "confidence": round(conf, 4),
            "entropy":    round(entropy, 4),
            "protocol":   f"0x{proto:02X}",
            "zone_x":     zone_x,
            "timestamp":  datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            "image_url":  "",
            "source":     "pi_daemon",
        })
        logger.info(f"Firebase push OK: {cls} ({conf:.2%})")
    except Exception as e:
        logger.error(f"Firebase push FAILED: {e}")

# ─── INFERENCE ───────────────────────────────────────────────────────────
def load_model(path: str):
    interpreter = Interpreter(model_path=path)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    return interpreter, input_details, output_details

def preprocess(image_array: np.ndarray, input_details) -> np.ndarray:
    from PIL import Image
    img = Image.fromarray(image_array).resize((INPUT_SIZE, INPUT_SIZE))
    arr = np.array(img, dtype=np.float32)  # raw [0-255], no normalization
    arr = np.expand_dims(arr, axis=0)

    if input_details[0]["dtype"] == np.int8:
        scale, zero_point = input_details[0]["quantization"]
        arr = (arr / scale + zero_point).astype(np.int8)
    elif input_details[0]["dtype"] == np.uint8:
        scale, zero_point = input_details[0]["quantization"]
        arr = (arr / scale + zero_point).astype(np.uint8)

    return arr

def run_inference(interpreter, input_details, output_details, image: np.ndarray):
    input_data = preprocess(image, input_details)
    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])

    if output_details[0]["dtype"] in (np.uint8, np.int8):
        scale, zero_point = output_details[0]["quantization"]
        output = (output.astype(np.float32) - zero_point) * scale

    probs = output[0]
    if np.any(probs < 0) or not (0.9 <= np.sum(probs) <= 1.1):
        exp_probs = np.exp(probs - np.max(probs))
        probs     = exp_probs / np.sum(exp_probs)

    # normalized entropy in [0,1] — 0 = certain, 1 = uniform across all classes
    p       = np.clip(probs, 1e-9, 1.0)
    entropy = float(-np.sum(p * np.log(p)) / np.log(len(CLASS_NAMES)))

    class_idx  = int(np.argmax(probs))
    confidence = float(probs[class_idx])
    return CLASS_NAMES[class_idx], confidence, entropy

# ─── MAIN LOOP ───────────────────────────────────────────────────────────
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-image", type=str, help="Path to test image (skips camera)")
    args = parser.parse_args()

    logger.info("="*60)
    logger.info("Pi Inference Daemon starting")
    logger.info("="*60)

    logger.info("Initializing Firebase...")
    cred = credentials.Certificate(FIREBASE_KEY)
    firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_DB})

    logger.info(f"Loading model from {MODEL_PATH}...")
    interpreter, input_det, output_det = load_model(MODEL_PATH)
    logger.info("Model loaded OK")

    logger.info(f"Opening UART: {UART_PORT} @ {UART_BAUD}")
    try:
        ser = serial.Serial(UART_PORT, UART_BAUD, timeout=1)
        time.sleep(0.1)
        ser.reset_input_buffer()
        logger.info("UART OK")
    except Exception as e:
        logger.warning(f"UART init failed: {e} — continuing without UART")
        ser = None

    test_image = None
    cam        = None
    if args.test_image:
        from PIL import Image
        logger.info(f"Test mode: using image {args.test_image}")
        test_image = np.array(Image.open(args.test_image).convert("RGB"))
    else:
        from picamera2 import Picamera2
        cam = Picamera2()
        cam.configure(cam.create_still_configuration(main={"size": (640, 480)}))
        cam.start()
        time.sleep(2)
        logger.info("Camera started")

    logger.info("Entering main loop...")
    inference_count = 0

    while not shutdown_flag:
        loop_start = time.monotonic()

        image = test_image if test_image is not None else cam.capture_array()

        t0 = time.monotonic()
        predicted_class, confidence, entropy = run_inference(
            interpreter, input_det, output_det, image)
        inference_ms = (time.monotonic() - t0) * 1000
        inference_count += 1

        logger.info(
            f"[#{inference_count}] Class: {predicted_class} | "
            f"Conf: {confidence:.2%} | Entropy: {entropy:.3f} | "
            f"Inference: {inference_ms:.0f}ms"
        )

        # Unknown fallback: low confidence OR smeared softmax (high entropy)
        threshold = CONFIDENCE_THRESHOLDS.get(predicted_class, 0.70)
        if confidence < threshold or entropy > ENTROPY_THRESHOLD:
            reason = "low conf" if confidence < threshold else "high entropy"
            logger.info(f"Flagging Unknown ({reason})")
            predicted_class = "Unknown"
            protocol_id     = 0xFF
        else:
            protocol_id = PROTOCOL_MAP[predicted_class]

        param1, param2 = PROTOCOL_PARAMS[protocol_id]
        zone_x         = 0x80
        packet         = build_packet(protocol_id, zone_x, param1, param2)

        if ser:
            send_packet(ser, packet)
        else:
            logger.info(f"UART skipped — packet would be: {packet.hex()}")

        push_to_firebase(predicted_class, confidence, entropy, protocol_id, zone_x)

        elapsed    = time.monotonic() - loop_start
        sleep_time = max(0, CAPTURE_INTERVAL - elapsed)
        if sleep_time > 0:
            time.sleep(sleep_time)

    logger.info("Shutting down...")
    if cam:
        cam.stop()
    if ser:
        ser.close()
    logger.info("Daemon stopped cleanly")

if __name__ == "__main__":
    main()
'''

# Save to Google Drive
with open('/kaggle/working/pi_daemon.py', 'w') as f:
    f.write(daemon_code)

print("pi_daemon.py saved to Google Drive!")
